In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%python
dbutils.widgets.text("catalogo", "catalog_gamd")
dbutils.widgets.text("storageName", "adlsgamd26")

# constantes para scope de nuestro datalake
scope = "accessScopeforDL"
key = "storageAccessKey"

In [0]:
%python
catalogo = dbutils.widgets.get("catalogo")
storageName = dbutils.widgets.get("storageName")

In [0]:
%python
spark.conf.set(f"fs.azure,account.key.{storageName}.dfs.core.windows.net", dbutils.secrets.get(scope = f"{scope}", key = f"{key}"))

In [0]:
%python

spark.catalog.setCurrentCatalog(f"{catalogo}")
print(spark.catalog.currentCatalog())

## DDLs Tablas

### Tablas Bronze

In [0]:
CREATE OR REPLACE TABLE ${catalogo}.bronze.aisles (
  aisle_id INT,
  aisle STRING
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/aisles";

CREATE OR REPLACE TABLE ${catalogo}.bronze.departments (
  department_id INT, department STRING
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/departments";

CREATE OR REPLACE TABLE ${catalogo}.bronze.order_products_prior (
  order_id INT, product_id INT, add_to_cart_order INT, reordered INT
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/order_products__prior";

CREATE OR REPLACE TABLE ${catalogo}.bronze.order_products_train (
  order_id INT, product_id INT, add_to_cart_order INT, reordered INT
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/order_products_train";

CREATE OR REPLACE TABLE ${catalogo}.bronze.orders (
  order_id INT, user_id INT, eval_set STRING, order_number INT, order_dow INT, order_hour_of_day INT, days_since_prior_order DOUBLE
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/orders";

CREATE OR REPLACE TABLE ${catalogo}.bronze.products (
  product_id INT, product_name STRING, aisle_id INT, department_id INT
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/products";

CREATE OR REPLACE TABLE ${catalogo}.bronze.sample_submission (
  order_id INT, products STRING
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/sample_submission";

### Creacion tablas Silver

In [0]:
CREATE OR REPLACE TABLE ${catalogo}.silver.order_products (
  order_id INT, product_id INT, add_to_cart_order INT,  reordered INT, id_eval_set int
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/order_products";

CREATE OR REPLACE TABLE ${catalogo}.silver.orders (
  order_id INT, user_id INT, id_eval_set INT, order_number INT, order_dow INT, order_hour_of_day INT, days_since_prior_order DOUBLE
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/orders";

CREATE OR REPLACE TABLE ${catalogo}.silver.cat_eval_set (
  id_eval_set INT, eval_set STRING
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/eval_set";

CREATE OR REPLACE TABLE ${catalogo}.silver.products_ordered (
  product_id INT, product_name STRING, aisle_id INT, aisle_name STRING, department_id INT, department STRING, id_eval_set INT, eval_set STRING, total_orders INT
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/products_ordered";

CREATE OR REPLACE TABLE ${catalogo}.silver.kpis_eval_set (
  id_eval_set INT, eval_set STRING, total_users INT, total_orders INT, avg_order_hour_of_day INT, avg_days_since_prior_order DOUBLE
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/kpis_eval_set";

### Creacion tablas Golden

In [0]:
CREATE OR REPLACE TABLE ${catalogo}.golden.kpis_products_ordered (
  product_id INT, product_name STRING, aisle_id INT, aisle_name STRING, department_id INT, department STRING, id_eval_set INT, eval_set STRING, total_orders INT
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/kpis_products_ordered";

CREATE OR REPLACE TABLE ${catalogo}.golden.kpis_eval_set (
  id_eval_set INT, eval_set STRING, total_users INT, total_orders INT, avg_order_hour_of_day INT, avg_days_since_prior_order DOUBLE
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/kpis_eval_set";